In [1]:
df_orders    = spark.read.table("silver_orders")
df_customers = spark.read.table("silver_customers")
df_products  = spark.read.table("silver_products")
df_sellers   = spark.read.table("silver_sellers")
df_payments  = spark.read.table("silver_payments")
df_reviews   = spark.read.table("silver_reviews")
df_order_items = spark.read.table("silver_order_items")
df_geolocation = spark.read.table("silver_geolocation")

StatementMeta(, 49de02cf-2cc3-483a-9b68-6af0a39f2e68, 3, Finished, Available, Finished, False)

In [2]:
from pyspark.sql.functions import col, sum as spark_sum, count, avg, round, year, month, countDistinct

StatementMeta(, 49de02cf-2cc3-483a-9b68-6af0a39f2e68, 4, Finished, Available, Finished, False)

joining orders and order_items to calculate revenue per order per month

EXPLANATION:
──────────────────────────────────────────────
JOIN orders + order_items
→ Orders alone has no prices
→ Order items alone has no dates
→ Joining gives us BOTH

filter("delivered")
→ Only count completed orders
→ Cancelled orders should not
  count as revenue

year() and month()
→ Extract year and month from timestamp
→ So we can group sales by month

countDistinct("order_id")
→ Counts unique orders per month
→ Not items - actual orders

count("order_item_id")
→ Counts total items sold
→ One order can have many items

In [3]:
df_orders_items = df_orders.join(df_order_items, on="order_id", how="inner")
gold_monthly_revenue = (
    df_orders_items 
    .filter(col("order_status") == "delivered")
    .withColumn("year", year(col("order_purchase_timestamp")))
    .withColumn("month", month(col("order_purchase_timestamp")))
    .groupBy("year", "month")
    .agg(
        round(spark_sum("total_item_value"), 2).alias("total_revenue"),
        countDistinct("order_id").alias("total_orders"),
        count("order_item_id").alias("total_items_sold"),
        round(avg("total_item_value"), 2).alias("avg_order_value")
    )
    .orderBy("year", "month")
)
gold_monthly_revenue.write.format("delta").mode("overwrite").saveAsTable("gold_monthly_revenue")

StatementMeta(, 49de02cf-2cc3-483a-9b68-6af0a39f2e68, 5, Finished, Available, Finished, False)

Who are our most valuable customers?
How much has each customer spent?
How many orders has each customer placed?
Where are our customers located?

In [4]:
from pyspark.sql.functions import col, sum as spark_sum, count, round, max as spark_max, min as spark_min, countDistinct, when

StatementMeta(, 49de02cf-2cc3-483a-9b68-6af0a39f2e68, 6, Finished, Available, Finished, False)

In [5]:
df_customer_orders = (
    df_orders
    .join(df_order_items,
          on="order_id",
          how="inner")
    .join(df_customers,
          on="customer_id",
          how="inner")
)


gold_customer_analysis = (
    df_customer_orders

   
    .filter(col("order_status") == "delivered")

    
    .groupBy(
        "customer_unique_id",
        "customer_city",
        "customer_state"
    )
    .agg(
        
        countDistinct("order_id")
        .alias("total_orders"),

       
        round(spark_sum("total_item_value"), 2)
        .alias("total_spent"),

        
        round(
            spark_sum("total_item_value") /
            countDistinct("order_id"), 2)
        .alias("avg_spend_per_order"),

       
        spark_min("order_purchase_timestamp")
        .alias("first_purchase_date"),
        spark_max("order_purchase_timestamp")
        .alias("last_purchase_date")
    )

   
    .withColumn("customer_tier",
        when(col("total_spent") >= 1000, 
             "High Value")
        .when(col("total_spent") >= 500,  
             "Mid Value")
        .otherwise("Low Value"))

    .orderBy(col("total_spent").desc())
)

gold_customer_analysis.write.format("delta").mode("overwrite").saveAsTable("gold_customer_analysis")

StatementMeta(, 49de02cf-2cc3-483a-9b68-6af0a39f2e68, 7, Finished, Available, Finished, False)

In [6]:
from pyspark.sql.functions import (
    col, sum as spark_sum, count,
    round, avg, countDistinct, desc
)

df_product_sales = (
    df_order_items
    .join(df_products,
          on="product_id",
          how="inner")
)

df_product_reviews = (
    df_reviews
    .groupBy("order_id")
    .agg(
        round(avg("review_score"), 2)
        .alias("avg_review_score")
    )
)

df_product_full = (
    df_product_sales
    .join(df_orders,
          on="order_id",
          how="inner")
    .join(df_product_reviews,
          on="order_id",
          how="left")
    .filter(col("order_status") == "delivered")
)

gold_product_performance = (
    df_product_full
    .groupBy(
        "product_id",
        "product_category_name_english"
    )
    .agg(
        # Total units sold
        count("order_item_id")
        .alias("units_sold"),

        # Total revenue generated
        round(spark_sum("total_item_value"), 2)
        .alias("total_revenue"),

        # Average selling price
        round(avg("price"), 2)
        .alias("avg_price"),

        # Average review score
        round(avg("avg_review_score"), 2)
        .alias("avg_review_score"),

        # Number of unique orders
        countDistinct("order_id")
        .alias("total_orders")
    )
    .orderBy(desc("total_revenue"))
)

gold_product_performance.write.format("delta").mode("overwrite").saveAsTable("gold_product_performance")

StatementMeta(, 49de02cf-2cc3-483a-9b68-6af0a39f2e68, 8, Finished, Available, Finished, False)

In [7]:
from pyspark.sql.functions import (
    col, sum as spark_sum, count,
    round, avg, countDistinct,
    when, desc
)

df_seller_sales = (
    df_order_items
    .join(df_sellers,
          on="seller_id",
          how="inner")
    .join(df_orders,
          on="order_id",
          how="inner")
    .filter(col("order_status") == "delivered")
)

df_seller_reviews = (
    df_reviews
    .groupBy("order_id")
    .agg(
        round(avg("review_score"), 2)
        .alias("avg_review_score")
    )
)

df_seller_full = (
    df_seller_sales
    .join(df_seller_reviews,
          on="order_id",
          how="left")
)

gold_seller_performance = (
    df_seller_full
    .groupBy(
        "seller_id",
        "seller_city",
        "seller_state"
    )
    .agg(
        
        round(spark_sum("total_item_value"), 2)
        .alias("total_revenue"),

        
        countDistinct("order_id")
        .alias("total_orders"),

        
        count("order_item_id")
        .alias("total_items_sold"),


        round(avg("avg_review_score"), 2)
        .alias("avg_review_score"),

    
        round(avg(
            col("order_delivered_customer_date")
            .cast("long") -
            col("order_purchase_timestamp")
            .cast("long")
        ) / 86400, 1)
        .alias("avg_delivery_days")
    )


    .withColumn("seller_tier",
        when(col("total_revenue") >= 50000,
             "Top Seller")
        .when(col("total_revenue") >= 10000,
             "Mid Seller")
        .otherwise("Regular Seller"))

    .orderBy(desc("total_revenue"))
)

gold_seller_performance.write.format("delta").mode("overwrite").saveAsTable("gold_seller_performance")

StatementMeta(, 49de02cf-2cc3-483a-9b68-6af0a39f2e68, 9, Finished, Available, Finished, False)

This is the MOST IMPORTANT table
It answers the ONE PAGE summary that
a CEO or executive wants to see:

→ Total revenue of the business
→ Total number of orders
→ Total number of customers
→ Average order value
→ Average review score
→ Total sellers on platform

In [10]:
from pyspark.sql.functions import (col, sum as spark_sum, count,round,avg,countDistinct, lit )

df_full = (
    df_orders
    .join(df_order_items,
          on="order_id",
          how="inner")
    .join(df_customers,
          on="customer_id",
          how="inner")
    .filter(col("order_status") == "delivered")
)

delivered_order_ids = df_orders \
    .filter(col("order_status") == "delivered") \
    .select("order_id")

avg_score = (
    df_reviews
    .join(delivered_order_ids,
          on="order_id",
          how="inner")
    .agg(round(avg("review_score"), 2)
         .alias("avg_review_score"))
    .first()["avg_review_score"]
)

total_sellers = (
    df_order_items
    .join(delivered_order_ids,
          on="order_id",
          how="inner")
    .agg(countDistinct("seller_id"))
    .first()[0]
)

gold_executive_kpis = (
    df_full
    .agg(
        round(spark_sum("total_item_value"), 2)
         .alias("total_revenue"),

        countDistinct("order_id")
         .alias("total_orders"),

        countDistinct("customer_unique_id")
         .alias("total_customers"),

        round(
            spark_sum("total_item_value") /
            countDistinct("order_id"), 2)
         .alias("avg_order_value"),

        count("order_item_id")
         .alias("total_items_sold")
    )
    .withColumn("avg_review_score",
        lit(avg_score))        

    .withColumn("total_sellers",
        lit(total_sellers))   
)

gold_executive_kpis.write.format("delta").mode("overwrite").saveAsTable("gold_executive_kpis")


StatementMeta(, 49de02cf-2cc3-483a-9b68-6af0a39f2e68, 12, Finished, Available, Finished, False)